# Kalamazoo County Treasury — Entity Resolution and Association Overview

## Purpose

This notebook provides a compact, reproducible overview of the **final** entity-resolution and association outputs produced by the project.

It replaces the historical development notebook that accumulated many one-off audits while the resolver was being designed.

The goals here are to:

- run the production pipeline components,
- summarize resolved entities and parcel roles,
- inspect association confidence and evidence at an aggregate level,
- perform lightweight integrity checks, and
- document how to interpret the outputs.

### Important interpretation rule

> Entity associations are **research leads**, not proof of legal ownership, beneficial ownership, control, common identity, fraud, or foreclosure risk.

The production Python modules and automated tests are the authoritative implementation and regression checks. This notebook is a readable analysis layer on top of those components.

## 1. Configuration

Update `DATA_FILE`, `SHEET_NAME`, and `TAX_YEAR` when analyzing a different annual Treasury export.

This notebook is designed to live in the repository's `notebooks/` folder. The setup cell automatically locates the project root, adds it to Python's import path, and resolves the source workbook relative to that root. It will also continue to work if Jupyter is started from the repository root or another subfolder within the repository.

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start=None):
    """Locate the repository root from the current notebook location."""
    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "etl_pipeline.py").is_file()
            and (candidate / "data_cleaning.py").is_file()
            and (candidate / "entity_resolution.py").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Start Jupyter from "
        "the repository or one of its subfolders."
    )


PROJECT_ROOT = find_project_root()

# Make the production project modules importable even when this
# notebook is opened from the notebooks/ subfolder.
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

from extraction import extract_data
from data_cleaning import DataCleaner
from entity_resolution import EntityResolver

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

DATA_FILE = PROJECT_ROOT / "2025 Roll - Parcels Export.xlsx"
SHEET_NAME = "Only Delinquent Parcels"
TAX_YEAR = 2025

if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Source workbook not found: {DATA_FILE}. "
        "Place the annual export in the project root or update DATA_FILE."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Data file: {DATA_FILE}")

Project root: C:\Users\mattp\Documents\kzoo_county_project
Data file: C:\Users\mattp\Documents\kzoo_county_project\2025 Roll - Parcels Export.xlsx


## 2. Run the production pipeline components

No entity-resolution logic is duplicated in this notebook. The notebook calls the same project classes used by the ETL pipeline.

In [2]:
extracted_df = extract_data(
    str(DATA_FILE),
    SHEET_NAME,
)

clean_df = DataCleaner().clean_data(
    extracted_df
)

(
    entity_occurrences,
    entities,
    parcel_entity_roles,
    association_evidence,
    entity_associations,
) = EntityResolver().build(
    clean_df
)

print(f"Tax year: {TAX_YEAR}")
print("Entity resolution complete.")

Tax year: 2025
Entity resolution complete.


## 3. Output overview

Each table has a distinct grain:

| Output | Grain |
|---|---|
| `clean_df` | one delinquent parcel |
| `entity_occurrences` | one resolved source-name occurrence |
| `entities` | one resolved entity |
| `parcel_entity_roles` | one entity/parcel/role relationship |
| `association_evidence` | one evidence record supporting an entity pair |
| `entity_associations` | one normalized entity pair |

In [3]:
output_summary = pd.DataFrame([
    ("Parcels", len(clean_df)),
    ("Source occurrences", entity_occurrences["source_occurrence_id"].nunique()),
    ("Entity occurrences", len(entity_occurrences)),
    ("Resolved entities", len(entities)),
    ("Parcel/entity roles", len(parcel_entity_roles)),
    ("Association evidence rows", len(association_evidence)),
    ("Entity associations", len(entity_associations)),
], columns=["output", "rows"])

output_summary

,output,rows
0,Parcels,3874
1,Source occurrences,10270
2,Entity occurrences,12067
3,Resolved entities,4954
4,Parcel/entity roles,12050
5,Association evidence rows,477
6,Entity associations,299


## 4. Resolved entity profile

In [4]:
entity_type_summary = (
    entities["entity_type"]
    .fillna("Unknown")
    .value_counts(dropna=False)
    .rename_axis("entity_type")
    .reset_index(name="entity_count")
)

entity_type_summary

,entity_type,entity_count
0,Person,4083
1,Organization,649
2,Joint,204
3,Unknown,18


In [5]:
occurrence_count_summary = pd.Series({
    "entities": len(entities),
    "entities_with_1_occurrence": (entities["occurrence_count"] == 1).sum(),
    "entities_with_2_or_more_occurrences": (entities["occurrence_count"] >= 2).sum(),
    "entities_with_5_or_more_occurrences": (entities["occurrence_count"] >= 5).sum(),
    "maximum_occurrence_count": entities["occurrence_count"].max(),
})

occurrence_count_summary.to_frame("value")

,value
entities,4954
entities_with_1_occurrence,1048
entities_with_2_or_more_occurrences,3906
entities_with_5_or_more_occurrences,260
maximum_occurrence_count,62


### Why occurrence counts need context

A high occurrence count means that multiple source occurrences resolved to the same entity. It is not a measure of ownership importance, risk, or legal control.

## 5. Parcel/entity roles

Roles preserve how each entity appeared in the source data instead of treating every appearance as ownership.

In [6]:
role_summary = (
    parcel_entity_roles["role"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="rows")
)

role_summary

,role,rows
0,Owner,4865
1,Tax Send To,4576
2,Taxpayer,2242
3,Owner Care Of,207
4,Taxpayer Care Of,160


In [7]:
role_entity_summary = (
    parcel_entity_roles
    .groupby("role")
    .agg(
        unique_entities=("entity_id", "nunique"),
        unique_parcels=("parcel_number", "nunique"),
    )
    .sort_values("unique_parcels", ascending=False)
)

role_entity_summary

,unique_entities,unique_parcels
role,,
Owner,4133,3874
Tax Send To,3843,3844
Taxpayer,1868,1979
Owner Care Of,188,202
Taxpayer Care Of,142,154


## 6. Association overview

Association confidence is based on the **strongest individual evidence rule** supporting a normalized entity pair.

Confidence is **not a probability** and should not be interpreted as a percentage likelihood that one entity owns, controls, or is identical to another.

In [8]:
confidence_summary = (
    entity_associations["association_confidence"]
    .value_counts(dropna=False)
    .rename_axis("association_confidence")
    .reset_index(name="association_count")
)

confidence_summary

,association_confidence,association_count
0,High,147
1,Medium,116
2,Low,36


In [9]:
evidence_depth_summary = (
    entity_associations["evidence_count"]
    .value_counts()
    .sort_index()
    .rename_axis("evidence_count")
    .reset_index(name="association_count")
)

evidence_depth_summary

,evidence_count,association_count
0,1,181
1,2,93
2,3,16
3,4,4
4,5,2
5,8,2
6,20,1


## 7. Evidence rules and strengths

Evidence remains separate from the summarized association so staff can understand **why** an association exists.

In [10]:
rule_summary = (
    association_evidence["evidence_rule"]
    .value_counts()
    .rename_axis("evidence_rule")
    .reset_index(name="evidence_rows")
)

rule_summary

,evidence_rule,evidence_rows
0,SHARED_OWNER_ADDRESS,91
1,SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER,79
2,SHARED_TAXPAYER_ADDRESS,63
3,SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER,55
4,SAME_PARCEL_CO_OWNER,49
5,SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO,49
6,SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER,49
7,SAME_PARCEL_ORG_OWNER_PERSON_OWNER_CARE_OF,42


In [11]:
strength_summary = (
    association_evidence["evidence_strength"]
    .value_counts(dropna=False)
    .rename_axis("evidence_strength")
    .reset_index(name="evidence_rows")
)

strength_summary

,evidence_strength,evidence_rows
0,Medium,252
1,High,183
2,Low,42


## 8. Lightweight integrity checks

These checks are intentionally generic so they remain useful for future annual datasets. Exact 2025 regression counts and reviewed cases belong in `tests/snapshot_2025/`.

In [12]:
assert clean_df["parcel_number"].is_unique
assert entities["entity_id"].is_unique
assert entity_occurrences["occurrence_id"].is_unique
assert association_evidence["evidence_id"].is_unique
assert entity_associations["association_id"].is_unique

valid_entity_ids = set(
    entities["entity_id"].astype(int)
)

assert set(
    parcel_entity_roles["entity_id"].astype(int)
).issubset(valid_entity_ids)

assert set(
    association_evidence["entity_id_a"].astype(int)
).issubset(valid_entity_ids)

assert set(
    association_evidence["entity_id_b"].astype(int)
).issubset(valid_entity_ids)

assert set(
    entity_associations["entity_id_a"].astype(int)
).issubset(valid_entity_ids)

assert set(
    entity_associations["entity_id_b"].astype(int)
).issubset(valid_entity_ids)

assert (
    entity_associations["entity_id_a"]
    < entity_associations["entity_id_b"]
).all()

print("Generic entity-resolution integrity checks passed.")

Generic entity-resolution integrity checks passed.


## 9. Evidence-summary consistency

For each normalized entity pair, the association's `evidence_count` should match the number of underlying evidence rows. Its `evidence_rule_count` should match the number of distinct evidence rules supporting that pair.

In [13]:
evidence_pair_summary = (
    association_evidence
    .groupby(
        ["entity_id_a", "entity_id_b"],
        as_index=False,
    )
    .agg(
        actual_evidence_count=("evidence_id", "size"),
        actual_evidence_rule_count=("evidence_rule", "nunique"),
    )
)

association_count_check = (
    entity_associations[
        [
            "association_id",
            "entity_id_a",
            "entity_id_b",
            "evidence_count",
            "evidence_rule_count",
        ]
    ]
    .merge(
        evidence_pair_summary,
        on=["entity_id_a", "entity_id_b"],
        how="left",
        validate="one_to_one",
    )
)

association_count_check["evidence_count_matches"] = (
    association_count_check["evidence_count"]
    == association_count_check["actual_evidence_count"]
)

association_count_check["rule_count_matches"] = (
    association_count_check["evidence_rule_count"]
    == association_count_check["actual_evidence_rule_count"]
)

assert association_count_check["evidence_count_matches"].all()
assert association_count_check["rule_count_matches"].all()

association_count_check[
    ["evidence_count_matches", "rule_count_matches"]
].value_counts()

evidence_count_matches  rule_count_matches
True                    True                  299
Name: count, dtype: int64

## 10. High-support associations — aggregate profile

To keep the repository notebook safe for sharing, this section does not display entity names, parcel numbers, or mailing addresses. Detailed case review should be performed against the secured database/API or local audit outputs.

In [14]:
high_support_profile = pd.Series({
    "associations_with_2_or_more_evidence_rows": (
        entity_associations["evidence_count"] >= 2
    ).sum(),
    "associations_with_3_or_more_evidence_rows": (
        entity_associations["evidence_count"] >= 3
    ).sum(),
    "maximum_evidence_rows_for_one_association": (
        entity_associations["evidence_count"].max()
    ),
})

high_support_profile.to_frame("value")

,value
associations_with_2_or_more_evidence_rows,118
associations_with_3_or_more_evidence_rows,25
maximum_evidence_rows_for_one_association,20


## 11. 2025 regression validation

For the exact validated 2025 snapshot, use the automated tests:

```powershell
python -m pytest tests/snapshot_2025 -v
```

The regression suite intentionally contains the frozen 2025 counts and reviewed cases. Keeping those checks in tests instead of duplicating them here reduces the risk that a notebook becomes a second, stale source of truth.

For the complete suite:

```powershell
python -m pytest -v
```

## 12. Interpretation and limitations

The final resolver is deliberately conservative.

- Source occurrences are preserved for traceability.
- Ambiguous records may remain separate rather than being aggressively merged.
- Associations retain explicit supporting evidence.
- Stronger evidence does not convert an association into a legal conclusion.
- Staff review and authoritative external sources remain necessary for consequential decisions.
- Administrative suppression/restoration exists so reviewed associations can be hidden from normal search results without deleting the audit history.

### Recommended use of this notebook

Use this notebook for:

- orientation,
- aggregate diagnostics,
- demonstrations of the final data model, and
- understanding how the major resolver outputs relate to one another.

Use the automated tests for regression assurance, the API/database for staff research, and the production Python modules for implementation details.

## 13. Project status

At handoff, the core system includes:

- annual extraction and cleaning,
- conservative entity resolution,
- parcel/entity role preservation,
- evidence-backed associations,
- MySQL persistence,
- read/search API endpoints,
- administrative suppression/restoration,
- snapshot tracking and annual refresh support,
- transactional refresh protection, and
- automated regression/contract/ETL tests.

Remaining work is primarily County deployment and integration rather than notebook-driven analysis.